# 02 · Errores sistemáticos: el modelo de tres términos

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/telmomm/nanovna-measurement-course/HEAD?labpath=es%2Fnotebooks%2F02_errores_sistematicos.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/telmomm/nanovna-measurement-course/blob/main/es/notebooks/02_errores_sistematicos.ipynb)

Un analizador vectorial de redes nunca te muestra el coeficiente de reflexión de tu dispositivo. Te muestra ese coeficiente de reflexión *después* de que haya pasado por las imperfecciones del propio instrumento. En este cuaderno construyes tú mismo un instrumento imperfecto, defecto a defecto, y observas qué le hace cada defecto a un dispositivo que conoces.

## Objetivos de aprendizaje

Al terminar este cuaderno serás capaz de:

- nombrar los tres términos de error sistemático de una medida de un puerto (directividad, desadaptación de fuente y seguimiento de reflexión) y decir de dónde viene cada uno;
- escribir la ecuación que liga el coeficiente de reflexión real $\Gamma$ con el medido $\Gamma_m$;
- predecir qué dispositivos distorsiona más cada término de error;
- reconocer en datos reales que una medida en bruto no es fiable.

**Necesitas:** la relación entre impedancia y coeficiente de reflexión, $\Gamma = (Z - Z_0)/(Z + Z_0)$, del cuaderno 01. Sin hardware.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/telmomm/nanovna-measurement-course

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from vnacourse import plotting
from vnacourse.data import load_repetitions
from vnacourse.errormodel import ErrorTerms, synthetic_error_terms
from vnacourse.reflection import gamma_to_z, z_to_gamma

plotting.use_course_style()

## 1. Tres maneras en que un instrumento distorsiona una reflexión

Para medir una reflexión, el instrumento envía una onda al dispositivo bajo prueba (DUT, *device under test*), separa la onda que vuelve y divide una por otra. Por el camino fallan tres cosas:

- **Directividad, $e_{00}$.** La parte que separa la onda de vuelta (un puente o un acoplador direccional) no es perfecta: un poco de la onda *enviada* se cuela directamente en el receptor sin llegar nunca al DUT. El instrumento ve una reflexión incluso cuando no la hay.
- **Desadaptación de fuente, $e_{11}$.** El puerto de prueba no es exactamente de 50 Ω. Parte de la onda reflejada por el DUT rebota en el puerto y viaja *otra vez* al DUT, y otra.
- **Seguimiento de reflexión, $e_{10}e_{01}$.** El camino hasta el DUT y de vuelta (cables, conectores, receptores) atenúa y retrasa la onda, de forma distinta a cada frecuencia.

Estos errores se llaman *sistemáticos* porque son los mismos cada vez que repites la medida. Eso es lo que permite eliminarlos. El ruido y un conector apretado de forma distinta cada vez son errores *aleatorios*; la calibración no los elimina.

La figura siguiente es la representación habitual del modelo: un instrumento perfecto a la izquierda, el DUT a la derecha y, entre ambos, un «adaptador de errores» que contiene los tres términos. Sigue las flechas desde *enviada* hasta *recibida*.

In [ ]:
plotting.draw_error_adapter(lang="es");

Hay dos maneras de ir de *enviada* a *recibida*:

1. directamente hacia abajo a través de $e_{00}$, sin tocar el DUT;
2. hasta el DUT, reflejarse ($\Gamma$) y volver a través de $e_{10}e_{01}$. Por el camino, la onda también puede recorrer el bucle DUT → $e_{11}$ → DUT cualquier número de veces, lo que multiplica el camino por $1 + e_{11}\Gamma + (e_{11}\Gamma)^2 + \dots = 1/(1 - e_{11}\Gamma)$.

Sumando ambos caminos se obtiene el **modelo de error de tres términos**:

$$\Gamma_m = e_{00} + \frac{e_{10}e_{01}\,\Gamma}{1 - e_{11}\,\Gamma}$$

Las cuatro magnitudes son números complejos, y los tres términos de error cambian con la frecuencia. Un instrumento perfecto tiene $e_{00} = 0$, $e_{11} = 0$ y $e_{10}e_{01} = 1$, y entonces $\Gamma_m = \Gamma$.

## 2. Construye tu propio instrumento imperfecto

El modelo es una línea de código. `terms` contiene los tres términos de error en el orden (directividad, desadaptación de fuente, seguimiento).

In [ ]:
def apply_error_model(gamma, terms):
    """Lo que muestra un instrumento imperfecto para una reflexión real `gamma`."""
    e00, e11, tracking = terms
    return e00 + tracking * gamma / (1 - e11 * gamma)


perfect = ErrorTerms(directivity=0, source_match=0, tracking=1)
print(apply_error_model(0.2, perfect))

Probamos el instrumento con cuatro dispositivos cuyo coeficiente de reflexión real conocemos exactamente:

| Dispositivo | Impedancia | $\Gamma$ real |
|---|---|---|
| cortocircuito (*short*) | 0 Ω | −1 |
| abierto (*open*) | ∞ | +1 |
| carga (*load*) | 50 Ω | 0 |
| resistencia | 75 Ω | +0.2 |

La función siguiente representa, para cada dispositivo, dónde está realmente en el plano $\Gamma$ y dónde dice el instrumento que está.

In [ ]:
devices = {
    "cortocircuito": -1.0,
    "abierto": 1.0,
    "carga": 0.0,
    "75 Ω": float(z_to_gamma(75).real),
}


def compare(terms, title):
    """Representa la reflexión real y la medida de los cuatro dispositivos de prueba."""
    fig, ax = plt.subplots(figsize=(5.2, 5.2))
    plotting.gamma_plane(ax, radius=1.4, title=title)
    # La carga y la resistencia de 75 Ω están muy cerca: separamos sus etiquetas.
    offsets = {"carga": (4, -16), "75 Ω": (-4, -16)}
    aligns = {"carga": "right", "75 Ω": "left"}
    for name, gamma in devices.items():
        measured = apply_error_model(gamma, terms)
        ax.annotate(
            "",
            xy=(measured.real, measured.imag),
            xytext=(gamma, 0),
            arrowprops={"arrowstyle": "-", "color": plotting.INK_SECONDARY, "linewidth": 0.8},
        )
        ax.plot(gamma, 0, "o", color=plotting.SERIES[0], markerfacecolor=plotting.SURFACE)
        ax.plot(measured.real, measured.imag, "o", color=plotting.SERIES[1])
        ax.annotate(
            name,
            (gamma, 0),
            xytext=offsets.get(name, (0, -16)),
            textcoords="offset points",
            ha=aligns.get(name, "center"),
        )
    ax.plot([], [], "o", color=plotting.SERIES[0], markerfacecolor=plotting.SURFACE, label="real")
    ax.plot([], [], "o", color=plotting.SERIES[1], label="medido")
    ax.legend(loc="upper left")
    return ax


compare(perfect, "Un instrumento perfecto: medido = real");

### 2.1 Solo directividad

Dale al instrumento un único defecto: una directividad de módulo 0.15.

> **Predice antes de ejecutar.** La carga de 50 Ω no refleja nada. ¿Qué mostrará este instrumento para ella? ¿Se desplazarán los cuatro dispositivos lo mismo o cantidades distintas?

In [ ]:
only_directivity = ErrorTerms(directivity=0.15 * np.exp(1j * np.pi / 3), source_match=0, tracking=1)
compare(only_directivity, "Solo directividad: todo se desplaza");

Todos los dispositivos se desplazan el mismo vector, $e_{00}$. El error absoluto es el mismo para todos, pero el error *relativo* no: el cortocircuito y el abierto (módulo 1) apenas lo notan, mientras que la carga, que debería leer exactamente 0, ahora lee 0.15. La directividad limita lo pequeña que puede ser una reflexión que quieras medir. Es el término que importa cuando quieres saber lo bien adaptado que está algo.

### 2.2 Solo seguimiento de reflexión

Ahora el único defecto es el camino de vuelta: pierde el 20 % de la amplitud y añade 40° de retraso de fase.

> **Predice antes de ejecutar.** ¿Qué dispositivo no se moverá en absoluto?

In [ ]:
only_tracking = ErrorTerms(
    directivity=0, source_match=0, tracking=0.8 * np.exp(-1j * np.radians(40))
)
compare(only_tracking, "Solo seguimiento: todo se escala y gira");

El seguimiento multiplica $\Gamma$: todo el plano se encoge y gira en torno al centro. La carga está en el centro, así que no se mueve. Los dispositivos alejados del centro son los que más se mueven. Si el módulo del seguimiento es *mayor* que 1, un cortocircuito o un abierto se leerán con $|\Gamma_m| > 1$, como si el dispositivo reflejara más de lo que recibe. Tenlo presente para la sección 4.

### 2.3 Solo desadaptación de fuente

El último defecto por separado: un puerto de prueba que no es de 50 Ω, con $|e_{11}| = 0.25$.

> **Predice antes de ejecutar.** Mira la ecuación. ¿Para qué dispositivo desaparece por completo $e_{11}$? ¿Y es el error del cortocircuito igual que el del abierto?

In [ ]:
only_match = ErrorTerms(directivity=0, source_match=0.25 * np.exp(1j * np.pi / 4), tracking=1)
compare(only_match, "Solo desadaptación de fuente: el error depende del dispositivo");

La desadaptación de fuente solo actúa a través del producto $e_{11}\Gamma$, así que necesita que exista una reflexión: la carga no se toca, la resistencia de 75 Ω se mueve un poco, y el cortocircuito y el abierto se mueven mucho, cada uno en una dirección *distinta*. Este es el término que hace que el modelo de error sea no lineal en $\Gamma$, y la razón por la que una corrección simple de «restar y dividir» no basta.

| Término | Efecto en el plano $\Gamma$ | Afecta más a |
|---|---|---|
| Directividad $e_{00}$ | desplaza todos los puntos el mismo vector | reflexiones pequeñas (dispositivos bien adaptados) |
| Seguimiento $e_{10}e_{01}$ | escala y gira en torno al centro | reflexiones grandes |
| Desadaptación de fuente $e_{11}$ | curva el plano; el error crece con $\lvert\Gamma\rvert$ | reflexiones grandes |

## 3. Los tres a la vez, a lo largo de la frecuencia

Un instrumento real tiene los tres defectos juntos, y cada uno cambia con la frecuencia. El paquete del curso proporciona un instrumento inventado, `synthetic_error_terms`, con errores exagerados para que todo se vea. Lo barremos de 50 kHz a 1.45 GHz, la misma banda que las medidas reales que usarás después.

In [ ]:
f_hz = np.linspace(50e3, 1.45e9, 401)
f_mhz = f_hz / 1e6
terms = synthetic_error_terms(f_hz)

fig, ax = plt.subplots()
ax.plot(f_mhz, np.abs(terms.directivity), label="directividad $|e_{00}|$")
ax.plot(f_mhz, np.abs(terms.source_match), label="desadaptación de fuente $|e_{11}|$")
ax.plot(f_mhz, np.abs(terms.tracking), label="seguimiento $|e_{10}e_{01}|$")
ax.set(xlabel="frecuencia (MHz)", ylabel="módulo", ylim=(0, 1.05))
ax.set_title("Términos de error del instrumento sintético")
ax.legend(loc="center right");

Los módulos son líneas suaves, casi planas. Las fases, que no se muestran, giran con la frecuencia porque cada camino dentro del instrumento tiene alguna longitud.

Conecta la resistencia de 75 Ω. Su coeficiente de reflexión real es 0.2 a todas las frecuencias: una línea plana.

> **Predice antes de ejecutar.** Los términos de error son suaves. ¿Será el $|\Gamma_m|$ medido también una línea suave, ligeramente desplazada?

In [ ]:
gamma_true = np.full(f_hz.shape, z_to_gamma(75))
gamma_measured = apply_error_model(gamma_true, terms)

fig, ax = plt.subplots()
ax.plot(f_mhz, np.abs(gamma_true), label="real")
ax.plot(f_mhz, np.abs(gamma_measured), label="medido")
ax.set(xlabel="frecuencia (MHz)", ylabel="$|\\Gamma|$", ylim=(0, 0.35))
ax.set_title("Resistencia de 75 Ω vista por el instrumento sintético")
ax.legend(loc="lower right");

La traza medida tiene un **rizado** que no está ni en el dispositivo ni en los módulos de los términos de error. Viene de las fases. $\Gamma_m$ es una suma de vectores complejos y, al cambiar la frecuencia, giran a ritmos distintos: donde se alinean la lectura es alta y donde se oponen es baja. El rizado en una traza que debería ser plana es la firma clásica de una medida sin calibrar (o mal calibrada).

En el plano $\Gamma$ el dispositivo real es un único punto, y la medida da vueltas a su alrededor:

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 5.2))
plotting.gamma_plane(ax, title="Resistencia de 75 Ω en la carta de Smith")
ax.plot(gamma_measured.real, gamma_measured.imag, color=plotting.SERIES[1], label="medido")
ax.plot(gamma_true[0].real, gamma_true[0].imag, "o", color=plotting.SERIES[0], label="real")
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.0));

Lo que normalmente quieres no es $\Gamma$ sino la impedancia. Convierte ambas trazas:

In [ ]:
z_measured = gamma_to_z(gamma_measured)

fig, ax = plt.subplots()
ax.plot(f_mhz, gamma_to_z(gamma_true).real, label="real")
ax.plot(f_mhz, z_measured.real, label="medido")
ax.set(xlabel="frecuencia (MHz)", ylabel="resistencia, Re Z (Ω)")
ax.set_title("Resistencia de la resistencia de 75 Ω según el instrumento")
ax.legend(loc="upper right")

print(f"Resistencia aparente: de {z_measured.real.min():.0f} Ω a {z_measured.real.max():.0f} Ω")
print(f"Reactancia aparente:  de {z_measured.imag.min():.0f} Ω a {z_measured.imag.max():.0f} Ω")

Una simple resistencia parece cambiar su resistencia con la frecuencia y tener una reactancia que no tiene. Aquí nada es ruido: ejecuta otra vez la celda y obtendrás exactamente la misma respuesta errónea. Un instrumento sin calibrar no es impreciso: es *sistemáticamente erróneo*.

La función que has escrito también está disponible como `vnacourse.errormodel.apply_error_model`, para que el siguiente cuaderno pueda reutilizarla. Comprueba que ambas coinciden:

In [ ]:
from vnacourse import errormodel

assert np.allclose(gamma_measured, errormodel.apply_error_model(gamma_true, terms))
print("El mismo resultado que vnacourse.errormodel.apply_error_model")

## 4. Un instrumento real

Hasta ahora todo era simulado. El repositorio incluye medidas reales tomadas con un NanoVNA-F V2: un abierto, un cortocircuito y una carga de 50 Ω, cada uno medido diez veces y guardado tal como lo mostró el instrumento. Los detalles están en [`data/README.md`](../data/README.md).

`load_repetitions` devuelve el eje de frecuencias y una fila por repetición. Promediamos las diez repeticiones de cada estándar.

> **Predice antes de ejecutar.** Si el instrumento fuera perfecto, ¿cuánto valdría $|\Gamma_m|$ para el abierto, el cortocircuito y la carga?

In [ ]:
f_real, open_raw = load_repetitions("standards", "open")
_, short_raw = load_repetitions("standards", "short")
_, load_raw = load_repetitions("standards", "load")
print(f"{open_raw.shape[0]} repeticiones, {open_raw.shape[1]} puntos de frecuencia")
print(f"de {f_real[0] / 1e3:.0f} kHz a {f_real[-1] / 1e9:.2f} GHz")

fig, ax = plt.subplots()
ax.plot(f_real / 1e6, np.abs(open_raw.mean(axis=0)), label="abierto")
ax.plot(f_real / 1e6, np.abs(short_raw.mean(axis=0)), label="cortocircuito")
ax.plot(f_real / 1e6, np.abs(load_raw.mean(axis=0)), label="carga")
ax.axhline(1, color=plotting.INK_SECONDARY, linewidth=0.8, linestyle="--")
ax.set(xlabel="frecuencia (MHz)", ylabel="$|\\Gamma_m|$")
ax.set_title("Lecturas en bruto de los tres estándares (NanoVNA-F V2)")
ax.legend(loc="upper right")

print(f"Mayor |Γm| en bruto del abierto: {np.abs(open_raw.mean(axis=0)).max():.2f}")

Por debajo de unos 150 MHz, el abierto y el cortocircuito leen claramente por encima de 1: según el instrumento, devuelven hasta el doble de la onda que reciben. Un circuito abierto y un cortocircuito son pasivos, así que no pueden hacerlo. La única explicación es el instrumento, y la sección 2.2 te dijo qué término produce esto.

Por encima de 150 MHz las tres trazas parecen casi ideales. «Casi» es la palabra importante: cuánto se alejan de lo ideal, y cómo eliminar lo que queda, es el tema del siguiente cuaderno.

## Comprueba lo que has entendido

1. Un instrumento tiene $e_{00} = 0.05$ y por lo demás es perfecto. Mides una antena y lees $|\Gamma_m| = 0.05$. ¿Qué puedes decir del $|\Gamma|$ real de la antena?

<details><summary>Respuesta</summary>

Solo que está entre 0 y 0.10. La lectura es la suma de la reflexión real y la directividad, y no conoces su fase relativa: pueden sumarse o cancelarse. Cuando la lectura es tan pequeña como la directividad, la medida no te dice casi nada.
</details>

2. ¿Por qué la carga de 50 Ω no se mueve en las secciones 2.2 y 2.3?

<details><summary>Respuesta</summary>

En el modelo, el seguimiento y la desadaptación de fuente multiplican a $\Gamma$. Con $\Gamma = 0$ no vuelve nada del DUT, así que no hay nada que escalar ni que volver a reflejar. Solo queda la directividad, que no involucra al DUT: para una carga perfecta, $\Gamma_m = e_{00}$.
</details>

3. Mides el mismo dispositivo cinco veces seguidas sin tocar nada y las cinco trazas se superponen perfectamente. ¿Significa eso que la medida es exacta?

<details><summary>Respuesta</summary>

No. Significa que la medida es *repetible*: los errores aleatorios son pequeños. Los errores sistemáticos son idénticos en las cinco trazas, así que esta prueba no los ve.
</details>

## Ejercicios

Las soluciones resueltas están en [`exercises/solutions/02_solutions.ipynb`](../exercises/solutions/02_solutions.ipynb).

1. **¿Qué término tiene la culpa?** Con el instrumento sintético de la sección 3, desactiva los términos de uno en uno (pon la directividad o la desadaptación de fuente a 0, o el seguimiento a 1) y representa cada vez el $|\Gamma_m|$ medido de la resistencia de 75 Ω. ¿Qué término contribuye más al rizado? ¿Cambia la respuesta si el dispositivo es un cortocircuito?
2. **Rizado y longitud.** En `synthetic_error_terms` la fase de cada término gira por un retardo. Construye tus propios `ErrorTerms` con una desadaptación de fuente $0.2\,e^{-j 2\pi f \tau}$ y prueba $\tau$ = 0.5 ns, 1 ns y 2 ns con un cortocircuito como dispositivo. ¿Cómo depende de $\tau$ la separación entre los picos del rizado? ¿Qué te dice eso sobre un rizado que veas en un instrumento real con un cable largo?
3. **El peor caso.** Para un instrumento con $|e_{00}| = 0.03$ y ningún otro error, calcula el intervalo de lecturas que podrías obtener para un dispositivo con $|\Gamma|$ real de 0.1. Expresa ambos extremos como pérdidas de retorno, $-20\log_{10}|\Gamma|$, en dB.
4. **Datos reales.** Representa la *fase* del abierto y del cortocircuito en bruto de la sección 4. Idealmente deberían ser 0° y 180° a todas las frecuencias. ¿Lo son? Superpón las diez repeticiones del abierto: ¿ves alguna diferencia entre ellas?

In [ ]:
# Tu trabajo aquí

## Resumen

- Una medida de un puerto está distorsionada por tres términos de error sistemático: directividad $e_{00}$, desadaptación de fuente $e_{11}$ y seguimiento de reflexión $e_{10}e_{01}$.
- Se combinan como $\Gamma_m = e_{00} + \dfrac{e_{10}e_{01}\,\Gamma}{1 - e_{11}\,\Gamma}$.
- La directividad perjudica a las reflexiones pequeñas; el seguimiento y la desadaptación de fuente, a las grandes.
- Como los términos son complejos y giran con la frecuencia, producen rizado y reactancia falsa, incluso en una simple resistencia.
- Los errores sistemáticos se repiten exactamente. Es una mala noticia (repetir la medida no ayuda) y una buena: tres incógnitas por frecuencia se pueden despejar.

**Siguiente:** en [03 · Calibración SOL](03_calibracion_sol.ipynb) mides tres dispositivos conocidos, despejas los tres términos de error y deshaces la distorsión, primero en este instrumento sintético y después en los datos reales.